# 02 — NumPy, Pandas & Datasets (Phases 1–2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/udaysharmadev/Ai-Roadmap/blob/main/notebooks/02_numpy_pandas.ipynb)

**Maps to:** `docs/machine-learning-roadmap.md > Phase 1 (math)` + `Phase 2 (data)`  
**Dataset:** `data/samples/titanic_sample.csv` (61 rows, synthetic, offline)  
**Milestone 2:** this notebook *is* the worked solution — "clean a custom CSV with Pandas".

Flow: vectors & stats → load & inspect → impute → encode/scale → visualize → one-call pipeline.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ai_roadmap.math_utils import (
    bayes_theorem,
    cosine_similarity,
    describe,
    linear_regression_closed_form,
    relu,
    sigmoid,
    softmax,
)
from ai_roadmap.data_wrangling import (
    basic_info,
    clean_dataframe,
    detect_outliers_iqr,
    encode_categoricals,
    impute_missing,
    load_csv,
    plot_correlation_heatmap,
    plot_numeric_hist,
    scale_numeric,
)

print(f"numpy {np.__version__} | pandas {pd.__version__}")

## 1. Vectors: dot, cosine, distance

Cosine similarity powers semantic search in Chunk 4 (RAG). Same function, tiny vectors here.

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 3.0, 4.0])
print(f"cosine(a, b) = {cosine_similarity(a, b):.4f}")  # ~0.9926
print(f"cosine(a, zeros) = {cosine_similarity(a, np.zeros(3))}")  # safe 0.0

# Why it matters: paraphrases score high even with different words (demo with toy vectors)
query = np.array([0.9, 0.1, 0.0])
doc_same_topic = np.array([0.8, 0.2, 0.1])
doc_other = np.array([0.0, 0.1, 0.9])
print(f"same-topic: {cosine_similarity(query, doc_same_topic):.3f}")
print(f"other-topic: {cosine_similarity(query, doc_other):.3f}")

## 2. Stats in one call: describe + Bayes + activations

In [ ]:
ages = np.array([3.7, 42.2, 40.2, 57.0, 22.0])
print(describe(ages))

# Bayes: P(disease|positive) with 1% prior, 90% sensitivity, 10.8% positive rate
print(f"posterior: {bayes_theorem(0.01, 0.9, 0.108):.4f}")

x = np.array([-2.0, -0.5, 0.0, 1.0, 3.0])
print(f"relu:    {relu(x)}")
print(f"sigmoid: {np.round(sigmoid(x), 3)}")
print(f"softmax: {np.round(softmax(np.array([2.0, 1.0, 0.1])), 3)}")  # sums to 1

# Closed-form regression sanity check: y = 2x
slope, intercept = linear_regression_closed_form(
    np.array([1.0, 2.0, 3.0]), np.array([2.0, 4.0, 6.0])
)
print(f"slope={slope:.2f} intercept={intercept:.2f}")

## 3. Load & inspect the Titanic sample

Note the deliberate mess: 6 missing ages, 3 missing embarked ports, 1 duplicate row — exactly what real data looks like.

In [ ]:
df = load_csv(ROOT / "data/samples/titanic_sample.csv")
print(df.shape)  # (61, 8) incl. duplicate
print(df.head(3).to_string())

info = basic_info(df)
print(f"shape: {info['shape']} | duplicated rows: {info['duplicated_rows']}")
print(f"missing per column: {info['missing']}")

## 4. Clean step-by-step (then compare with the one-call pipeline)

In [ ]:
# 4a. Impute: mean age, mode embarked — input df is never mutated
imputed = impute_missing(df)
print(f"missing before: {int(df.isna().sum().sum())} -> after: {int(imputed.isna().sum().sum())}")

# 4b. Outliers in fare (1.5xIQR rule)
mask = detect_outliers_iqr(imputed, "fare")
print(f"fare outliers: {int(mask.sum())} rows, e.g. {imputed.loc[mask, 'fare'].head(3).tolist()}")

# 4c. Encode + scale a copy (keep original labels for modeling in Chunk 3)
encoded = encode_categoricals(imputed, columns=["sex", "embarked"])
print(f"encoded columns: {list(encoded.columns)}")
scaled = scale_numeric(imputed, columns=["age", "fare"], method="standard")
print(f"scaled age mean~{scaled['age'].mean():.3f} std~{scaled['age'].std():.3f}")

## 5. Visualize (figures saved headless — no GUI needed)

In [ ]:
out = ROOT / "outputs"
out.mkdir(exist_ok=True)

fig1 = plot_numeric_hist(imputed, "age")
fig1.savefig(out / "titanic_age_hist.png", dpi=120)
plt.close(fig1)

fig2 = plot_correlation_heatmap(imputed[["pclass", "age", "sibsp", "fare", "survived"]])
fig2.savefig(out / "titanic_corr.png", dpi=120)
plt.close(fig2)

print(f"saved: {out / 'titanic_age_hist.png'}")
print(f"saved: {out / 'titanic_corr.png'}")

## 6. The one-call Milestone-2 pipeline ✅

This is what your own wrangling script should look like: dedupe → impute → report.

In [ ]:
clean, report = clean_dataframe(df)
print(report)
print(clean.head(3).to_string())

assert report["duplicates_removed"] == 1, "expected the 1 deliberate duplicate removed"
assert all(v == 0 for v in report["missing_after"].values()), "all NaNs must be gone"
print("\nMilestone 2 checks passed ✅")

## ✅ Exercises

1. Re-run `clean_dataframe(df, scale='standard', one_hot=True)` — how does the shape change?
2. Try `housing_sample.csv` through the same pipeline. Which columns need encoding?
3. Change `numeric_strategy='median'` — does the mean age shift?

**Next (Chunk 3):** `03_sklearn_end_to_end.ipynb` — train a classifier on this cleaned frame.